# 01 - Data Loading & Graph Construction

**Mục tiêu của notebook này:**
- Load `USvideos.csv` + `US_category_id.json`
- Làm sạch dữ liệu
- Xây dựng weighted graph (node = channel, edge = co-trending)
- Export graph để dùng ở notebook `02_analysis.ipynb` và `03_models.ipynb`


In [ ]:
import pandas as pd
import json
import networkx as nx
from itertools import combinations
from pathlib import Path
import pickle

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../output")
(OUTPUT_DIR / "graphs").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "tables").mkdir(parents=True, exist_ok=True)

COUNTRY = "US"  # đổi thành "GB" khi chạy cho dataset Anh


## 1. Load dữ liệu

In [ ]:
csv_path = DATA_DIR / f"{COUNTRY}videos.csv"
json_path = DATA_DIR / f"{COUNTRY}_category_id.json"

df = pd.read_csv(csv_path, encoding="utf-8", on_bad_lines="skip")
print(f"Số dòng ban đầu: {len(df):,}")
df.head()


In [ ]:
with open(json_path, "r", encoding="utf-8") as f:
    category_data = json.load(f)

category_map = {
    int(item["id"]): item["snippet"]["title"]
    for item in category_data["items"]
}
print(f"Số category: {len(category_map)}")
category_map


## 2. Làm sạch dữ liệu

In [ ]:
# Loại bỏ video lỗi / đã bị xóa
if "video_error_or_removed" in df.columns:
    before = len(df)
    df = df[df["video_error_or_removed"] == False].copy()
    print(f"Loại {before - len(df):,} dòng video_error_or_removed=True")

# Loại bỏ dòng thiếu thông tin quan trọng
df = df.dropna(subset=["channel_title", "category_id", "trending_date"]).copy()

# Map category_id -> tên category
df["category_id"] = df["category_id"].astype(int)
df["category_name"] = df["category_id"].map(category_map)
df = df.dropna(subset=["category_name"]).copy()

# Chuẩn hóa trending_date (format gốc thường là YY.DD.MM)
df["trending_date"] = pd.to_datetime(df["trending_date"], format="%y.%d.%m", errors="coerce")
df = df.dropna(subset=["trending_date"]).copy()

print(f"Số dòng sau khi làm sạch: {len(df):,}")
print(f"Số channel duy nhất: {df['channel_title'].nunique():,}")


In [ ]:
# Loại bỏ video trùng lặp trong cùng 1 ngày (đề phòng crawl lặp)
df = df.drop_duplicates(subset=["video_id", "trending_date"]).copy()
print(f"Số dòng sau khi loại trùng: {len(df):,}")


In [ ]:
# Lưu lại dữ liệu đã làm sạch để dùng ở các bước sau
df.to_csv(OUTPUT_DIR / "tables" / f"{COUNTRY}_cleaned.csv", index=False)
df.shape


## 3. Xây dựng graph (node = channel, edge = co-trending)

> **Lưu ý phương pháp:** với mỗi `(category_id, trending_date)`, tất cả channel cùng trending sẽ tạo thành một **clique đầy đủ**. Đây là giới hạn cần nêu rõ khi diễn giải clustering coefficient và average shortest path ở notebook `02_analysis.ipynb`.

In [ ]:
def build_cotrending_graph(df: pd.DataFrame) -> nx.Graph:
    """
    Xây weighted graph:
    - node = channel_title
    - edge = 2 channel cùng trending trong cùng category_id + trending_date
    - weight = số lần co-trending
    """
    G = nx.Graph()
    G.add_nodes_from(df["channel_title"].unique())

    grouped = df.groupby(["category_id", "trending_date"])["channel_title"].apply(
        lambda x: sorted(set(x))
    )

    for channels in grouped:
        if len(channels) < 2:
            continue
        for u, v in combinations(channels, 2):
            if G.has_edge(u, v):
                G[u][v]["weight"] += 1
            else:
                G.add_edge(u, v, weight=1)

    return G

G = build_cotrending_graph(df)
print(f"Số node: {G.number_of_nodes():,}")
print(f"Số edge: {G.number_of_edges():,}")


## 4. Thông tin cơ bản của graph

In [ ]:
degrees = [d for _, d in G.degree()]
print(f"Degree trung bình: {sum(degrees)/len(degrees):.2f}")
print(f"Degree lớn nhất: {max(degrees)}")
print(f"Density: {nx.density(G):.6f}")

# Cần cho average_shortest_path_length / diameter ở notebook 02
print(f"Số connected component: {nx.number_connected_components(G)}")
largest_cc = max(nx.connected_components(G), key=len)
print(f"Kích thước connected component lớn nhất: {len(largest_cc):,} / {G.number_of_nodes():,} node")


## 5. Export graph

In [ ]:
# Lưu graph dạng pickle (giữ đầy đủ thuộc tính) để notebook 02, 03 load lại
with open(OUTPUT_DIR / "graphs" / f"{COUNTRY}_graph.pickle", "wb") as f:
    pickle.dump(G, f)

# Lưu thêm dạng GraphML để mở bằng Gephi nếu cần
nx.write_graphml(G, OUTPUT_DIR / "graphs" / f"{COUNTRY}_graph.graphml")

print("Đã export graph thành công.")


## Tiếp theo

Dùng file `output/graphs/{COUNTRY}_graph.pickle` ở notebook `02_analysis.ipynb` để tính các network metrics (degree distribution, power-law, centrality, community detection).